# AT-TPC Latent Space Exploration
---
This notebook explores AT-TPC event embeddings with k-means, PCA, UMAP, t-SNE, and a silhouette score.

In [ ]:
import os
import sys
import numpy as np
sys.path.append('../../')
from clustering import t_SNE_clustering
from clustering import UMAP_embedding
from clustering import k_means_clustering
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
# create a folder for results of global feature exploration

folder_path = './plots/exploration'
if not os.path.exists(folder_path):
  os.makedirs(folder_path, exist_ok=True)

## 1. Load Data
---
Set the feature and label paths, then load the `.npy` files. The notebook keeps data paths and plot labels in one place so the clustering calls stay simple.

`class_names` is optional. Leave it as `None` to use numeric labels in plot legends, or provide one name for each grouped class.



In [ ]:
# Set file paths
features_path = '../../data/features.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED
labels_path = '../../data/master_labels.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED

# Optional plot labels. Use None for simple numeric labels.
# If provided, include one name for each grouped label.
class_names = None
# class_names = ["0,1,2 Tracks", "3 Tracks", "4, 5 Tracks"]

features = np.load(features_path)
print(f"Features loaded successfully! Shape: {features.shape}")

In [ ]:
# Load labels
label_data = np.load(labels_path)

In [ ]:
# Convert labels to a flat integer array
labels = np.asarray(label_data).astype(int).ravel()

# Group track labels into 3 classes
labels = np.where(np.isin(labels, [0, 1, 2]), 0, labels)
labels = np.where(np.isin(labels, [3]), 1, labels)
labels = np.where(np.isin(labels, [4, 5]), 2, labels)

features = StandardScaler().fit_transform(features)

print(f"Data scaled successfully! {features.shape}")

## 2. K-Means
---
Run k-means on the scaled embeddings. The clusters can then be compared with the grouped labels, using `class_names` for legend text when it is provided.

In [ ]:
save_dir = './plots/k_means'

features_glob, cluster_labels, cluster_indices = k_means_clustering(
    features,
    labels,
    dimension=2,
    save_dir=save_dir,
    num_samples_to_print=10,
    class_names=class_names
)

## 3. PCA Projection
---
Use PCA as a simple baseline before UMAP and t-SNE. PCA can project the scaled embeddings into either 2D or 3D by changing `dimension`. It shows the main directions of variation and uses the same labels and optional `class_names` as the other plots.

In [ ]:
from clustering import pca_clustering

save_dir = './plots/pca'

# Use dimension=2 for a 2D plot or dimension=3 for a 3D plot.
pca_results = pca_clustering(
    features,
    labels,
    dimension=2,
    save_dir=save_dir,
    class_names=class_names
)

## 4. Map Clusters Back to Rows
---
Record which feature rows were sampled from each k-means cluster.

In [ ]:
def transform_ids(cluster_ids, original_ids):
    original = []

    for index in cluster_ids:
        original.append(original_ids[index])
    return original

sequential_row_ids = np.arange(len(features))
clusters = []

print("Transformed indices:")
for i, cluster in enumerate(cluster_indices):
    transformed_cluster = transform_ids(cluster, sequential_row_ids)
    clusters.append(transformed_cluster)
    print(f"Cluster {i}: {transformed_cluster}")

## 5. Check Cluster Labels
---
Print the true labels for the sampled events in each cluster.

In [ ]:
def true_labels(indices, labels):
    c_labels = []
    for index in indices:
        c_labels.append(labels[index])

    return c_labels

print("True labels verification per cluster:")
for i, cluster in enumerate(cluster_indices):
    print(f"Cluster {i}: {true_labels(cluster, labels)}")

## 6. UMAP Visualization
---
Run UMAP on the scaled feature matrix and color the 2D plot by label. The plot uses numeric labels unless `class_names` is set, saves under `./plots/umap/`, and stores the coordinates in `umap_results`.

In [ ]:
neighbors = 50
plt_colors = ['blue', 'green', 'red']

fig, ax = plt.subplots(figsize=(10, 7))

umap_results = UMAP_embedding(
    features,
    dimension=2,
    ax=ax,
    labels=labels,
    neighbors=neighbors,
    class_names=class_names,
    plt_colors=plt_colors,
    save_dir='./plots'
)

ax.legend()
plt.title(f"UMAP 2D Manifold Embedding (n_neighbors={neighbors})")
plt.savefig('./plots/umap/umap_unified_manifold.png', dpi=200)
plt.show()

## 7. t-SNE Visualization
---
Run t-SNE on the scaled feature matrix and color the 2D plot by label. The plot uses numeric labels unless `class_names` is set. t-SNE is useful for local structure, but far-apart distances should be interpreted carefully. The plot is saved under `./plots/t-sne/`, and the coordinates are stored in `tsne_results`.

In [ ]:
perplexity = 40
plt_colors = ['blue', 'green', 'red']

fig, ax = plt.subplots(figsize=(10, 7))

tsne_results = t_SNE_clustering(
    features,
    dimension=2,
    ax=ax,
    labels=labels,
    perplexity=perplexity,
    class_names=class_names,
    plt_colors=plt_colors,
    save_dir='./plots'
)

ax.legend()
plt.title(f"t-SNE 2D Manifold Embedding (perplexity={perplexity})")
plt.savefig('./plots/t-sne/tsne_unified_manifold.png', dpi=200)
plt.show()

## 8. Silhouette Score
---
Compute a silhouette score on the scaled features using the grouped labels. This gives a simple score for class separation.

A score near **1.0** means strong separation. A score near **0.0** means weak separation.

In [ ]:
from sklearn.metrics import silhouette_score

# Compute on the scaled feature space, not the 2D projections
# sample_size avoids expensive full computation at high dimensionality
sil_score = silhouette_score(features, labels, sample_size=2000, random_state=42)

print("=" * 50)
print(f"Silhouette Score (scaled feature space): {sil_score:.4f}")
print("=" * 50)

if sil_score >= 0.7:
    print("Interpretation: Strong, well-separated cluster structure")
elif sil_score >= 0.5:
    print("Interpretation: Reasonable cluster structure")
elif sil_score >= 0.25:
    print("Interpretation: Weak cluster structure — representations may need improvement")
else:
    print("Interpretation: No meaningful separation — contrastive objective needs tuning")